# Sign Language Recognition



# 1. Prepare Dataset

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install -U tf_keras
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import zipfile
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.utils import to_categorical

zip_file_path = '/content/drive/MyDrive/slmnist.zip'
extract_destination = 'SignLanguage/'

with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extract_destination)

train_csv = os.path.join(extract_destination, 'sign_mnist_train.csv')
test_csv  = os.path.join(extract_destination, 'sign_mnist_test.csv')

train_df = pd.read_csv(train_csv)
test_df = pd.read_csv(test_csv)

# Sign Language MNIST labels:
# A=0, B=1, C=2, ... but J is skipped in the original dataset
selected_label_ids = [0, 1]   # A and B
class_names = ['A', 'B']

train_df = train_df[train_df['label'].isin(selected_label_ids)]
test_df = test_df[test_df['label'].isin(selected_label_ids)]

def csv_to_images_and_labels(df):
    labels = df['label'].values
    pixels = df.drop('label', axis=1).values

    # Original images are 28x28 grayscale
    images = pixels.reshape(-1, 28, 28, 1).astype('float32')

    # Resize to 96x96 for Arduino camera/model compatibility
    images = tf.image.resize(images, [96, 96]).numpy()

    # Normalize to 0-1
    images = images / 255.0

    # Remap labels from original IDs to 0,1
    label_map = {0: 0, 1: 1}
    labels = np.array([label_map[label] for label in labels])

    labels = to_categorical(labels, num_classes=2)

    return images, labels

x_train, y_train = csv_to_images_and_labels(train_df)
x_test, y_test = csv_to_images_and_labels(test_df)

print(x_train.shape)
print(y_train.shape)
print(x_test.shape)
print(y_test.shape)

(2136, 96, 96, 1)
(2136, 2)
(763, 96, 96, 1)
(763, 2)


# 2. Create Model


In [3]:
from tensorflow.keras import layers, models

### DEFINE THE MODEL

model = models.Sequential()

# Input Layer
model.add(layers.Input(shape=(96, 96, 1)))

# Convolutional Block 1
model.add(layers.Conv2D(8, (3, 3), activation='relu', padding='same'))
model.add(layers.MaxPooling2D((2, 2)))  # Reduces size by half

# Convolutional Block 2
model.add(layers.Conv2D(16, (3, 3), activation='relu', padding='same'))
model.add(layers.MaxPooling2D((2, 2)))  # Further reduces size by half

# Convolutional Block 3
model.add(layers.Conv2D(32, (3, 3), activation='relu', padding='same'))
model.add(layers.MaxPooling2D((2, 2)))  # Further reduces size by half

# Convolutional Block 4
model.add(layers.Conv2D(64, (3, 3), activation='relu', padding='same'))
model.add(layers.MaxPooling2D((2, 2)))  # Further reduces size by half

# Flatten Layer
model.add(layers.Flatten())

# Fully connected layer

model.add(layers.Dense(32, activation='relu'))

# Output Layer
model.add(layers.Dense(2, activation='softmax'))


### COMPILE THE MODEL

model.compile(loss='categorical_crossentropy',
              optimizer='adam',
              metrics=['acc'])

model.summary()


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 96, 96, 8)         80        
                                                                 
 max_pooling2d (MaxPooling2  (None, 48, 48, 8)         0         
 D)                                                              
                                                                 
 conv2d_1 (Conv2D)           (None, 48, 48, 16)        1168      
                                                                 
 max_pooling2d_1 (MaxPoolin  (None, 24, 24, 16)        0         
 g2D)                                                            
                                                                 
 conv2d_2 (Conv2D)           (None, 24, 24, 32)        4640      
                                                                 
 max_pooling2d_2 (MaxPoolin  (None, 12, 12, 32)        0

In [4]:
import pandas as pd
import numpy as np
import tensorflow as tf

image_size = (96, 96)
batch_size = 32

train_csv = '/content/SignLanguage/sign_mnist_train.csv'
test_csv  = '/content/SignLanguage/sign_mnist_test.csv'

train_df = pd.read_csv(train_csv)
test_df = pd.read_csv(test_csv)

# A = 0, B = 1
selected_label_ids = [0, 1]
label_map = {0: 0, 1: 1}

train_df = train_df[train_df['label'].isin(selected_label_ids)]
test_df = test_df[test_df['label'].isin(selected_label_ids)]

def csv_to_dataset(df, shuffle=False, augment=False):
    labels = df['label'].values
    pixels = df.drop('label', axis=1).values

    images = pixels.reshape(-1, 28, 28, 1).astype('float32')
    images = tf.image.resize(images, image_size)
    images = images / 255.0

    labels = np.array([label_map[x] for x in labels])
    labels = tf.keras.utils.to_categorical(labels, num_classes=2)

    dataset = tf.data.Dataset.from_tensor_slices((images, labels))

    if shuffle:
        dataset = dataset.shuffle(buffer_size=len(images))

    if augment:
        data_augmentation = tf.keras.Sequential([
            tf.keras.layers.RandomRotation(0.05),
            tf.keras.layers.RandomTranslation(0.10, 0.10),
            tf.keras.layers.RandomZoom(0.10),
        ])

        dataset = dataset.map(
            lambda x, y: (data_augmentation(x, training=True), y),
            num_parallel_calls=tf.data.AUTOTUNE
        )

    dataset = dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return dataset

train_generator = csv_to_dataset(train_df, shuffle=True, augment=True)
test_generator = csv_to_dataset(test_df, shuffle=False, augment=False)

print("Train batches:", len(train_generator))
print("Test batches:", len(test_generator))

Train batches: 67
Test batches: 24


In [5]:
# Train the model with 150 epochs - if skipping this xfer learning model, comment this cell out.
EPOCHS = 50
BATCH_SIZE = 32

history = model.fit(
    x_train,
    y_train,
    validation_data=(x_test, y_test),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE
)

Epoch 1/50
67/67 [==============================] - 10s 40ms/step - loss: 0.3551 - acc: 0.8184 - val_loss: 0.0032 - val_acc: 1.0000
Epoch 2/50
67/67 [==============================] - 1s 10ms/step - loss: 0.0019 - acc: 1.0000 - val_loss: 9.0948e-05 - val_acc: 1.0000
Epoch 3/50
67/67 [==============================] - 1s 11ms/step - loss: 1.9856e-04 - acc: 1.0000 - val_loss: 2.5368e-05 - val_acc: 1.0000
Epoch 4/50
67/67 [==============================] - 1s 11ms/step - loss: 8.1301e-05 - acc: 1.0000 - val_loss: 1.7483e-05 - val_acc: 1.0000
Epoch 5/50
67/67 [==============================] - 1s 10ms/step - loss: 4.7395e-05 - acc: 1.0000 - val_loss: 7.6148e-06 - val_acc: 1.0000
Epoch 6/50
67/67 [==============================] - 1s 10ms/step - loss: 3.2227e-05 - acc: 1.0000 - val_loss: 7.3738e-06 - val_acc: 1.0000
Epoch 7/50
67/67 [==============================] - 1s 8ms/step - loss: 2.4418e-05 - acc: 1.0000 - val_loss: 4.4188e-06 - val_acc: 1.0000
Epoch 8/50
67/67 [=====================

#3. Convert to TensorFlow Lite

In [6]:
import numpy as np


# Save Trained Model  - this is crashing
sign_ab_model = "sign_ab_model"
tf.saved_model.save(model, sign_ab_model)


# Convert Model using TFLiteConverter

#FER_SAVED_MODEL = '/content/fer_saved_model'
import pathlib
converter = tf.lite.TFLiteConverter.from_saved_model(sign_ab_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8
def representative_dataset_gen():
    for _ in range(100):
        # Get sample input data as a numpy array
        yield [np.random.rand(1, 96, 96, 1).astype(np.float32)]

converter.representative_dataset = representative_dataset_gen

#following line is crashing the runtime
tflite_model = converter.convert()
tflite_models_dir = pathlib.Path("/content/")

tflite_model_file = tflite_models_dir/'sign_ab_model.tflite'
tflite_model_file.write_bytes(tflite_model)
# This will report back the file size in bytes

108120

In [7]:
from tqdm import tqdm
import numpy as np
import pandas as pd
import tensorflow as tf

# Load TFLite model and allocate tensors
interpreter = tf.lite.Interpreter(model_path='/content/sign_ab_model.tflite')
interpreter.allocate_tensors()

# Load Sign Language MNIST CSV test file
test_csv_path = '/content/SignLanguage/sign_mnist_test.csv'
test_df = pd.read_csv(test_csv_path)

# Select only A and B
# A = 0, B = 1
selected_label_ids = [0, 1]
label_map = {0: 0, 1: 1}

test_df = test_df[test_df['label'].isin(selected_label_ids)]

# Separate labels and pixels
test_labels_raw = test_df['label'].values
test_pixels = test_df.drop('label', axis=1).values

# Convert CSV rows to 28x28 grayscale images
test_images = test_pixels.reshape(-1, 28, 28, 1).astype(np.float32)

# Resize to 96x96 to match the model input
test_images = tf.image.resize(test_images, [96, 96]).numpy()

# Remap labels to 0 and 1
test_labels = np.array([label_map[label] for label in test_labels_raw])

# Check input and output tensor details
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

input_index = input_details[0]['index']
output_index = output_details[0]['index']

print('Input Details:', input_details)
print('Output Details:', output_details)

input_shape = input_details[0]['shape']
input_dtype = input_details[0]['dtype']

print('Expected Input Shape:', input_shape)
print('Expected Input Dtype:', input_dtype)

predictions = []
test_imgs = []

def preprocess_image(image):
    image = image.astype(np.float32)

    # Match your previous Arduino-style normalization:
    # 0..255 -> -1..1 -> int8
    normalized_image = (image / 255.0) * 2.0 - 1.0
    int8_image = np.clip(normalized_image * 127, -128, 127).astype(np.int8)

    return int8_image

# Limit to first 100 examples
num_examples = min(100, len(test_images))

for i in tqdm(range(num_examples)):
    img = test_images[i]
    label = test_labels[i]

    preprocessed_img = preprocess_image(img)

    # Add batch dimension: [96, 96, 1] -> [1, 96, 96, 1]
    preprocessed_img = np.expand_dims(preprocessed_img, axis=0)

    interpreter.set_tensor(input_index, preprocessed_img)
    interpreter.invoke()

    predictions.append(interpreter.get_tensor(output_index))
    test_imgs.append(img)

print(len(predictions))

# Evaluate accuracy
score = 0

for i in range(len(predictions)):
    prediction = np.argmax(predictions[i])
    label = test_labels[i]

    if prediction == label:
        score += 1

print(f"Out of {num_examples} predictions, {score} were correct")
print(f"Accuracy: {score / num_examples:.2%}")

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


Input Details: [{'name': 'serving_default_input_1:0', 'index': 0, 'shape': array([ 1, 96, 96,  1], dtype=int32), 'shape_signature': array([-1, 96, 96,  1], dtype=int32), 'dtype': <class 'numpy.int8'>, 'quantization': (0.00392156234011054, -128), 'quantization_parameters': {'scales': array([0.00392156], dtype=float32), 'zero_points': array([-128], dtype=int32), 'quantized_dimension': 0, 'block_size': 0}, 'sparsity_parameters': {}}]
Output Details: [{'name': 'StatefulPartitionedCall:0', 'index': 25, 'shape': array([1, 2], dtype=int32), 'shape_signature': array([-1,  2], dtype=int32), 'dtype': <class 'numpy.int8'>, 'quantization': (0.00390625, -128), 'quantization_parameters': {'scales': array([0.00390625], dtype=float32), 'zero_points': array([-128], dtype=int32), 'quantized_dimension': 0, 'block_size': 0}, 'sparsity_parameters': {}}]
Expected Input Shape: [ 1 96 96  1]
Expected Input Dtype: <class 'numpy.int8'>


100%|██████████| 100/100 [00:00<00:00, 799.63it/s]

100
Out of 100 predictions, 100 were correct
Accuracy: 100.00%


#4. Get Byte Array

In [31]:
MODEL_TFLITE = '/content/sign_ab_model.tflite'
MODEL_TFLITE_MICRO = 'sign_ab_model.cc'
!xxd -i {MODEL_TFLITE} > {MODEL_TFLITE_MICRO}
REPLACE_TEXT = MODEL_TFLITE.replace('/', '_').replace('.', '_')
!sed -i 's/'{REPLACE_TEXT}'/g_model/g' {MODEL_TFLITE_MICRO}
